# Hybrid Architecture Decision Report

A hybrid approach combines retrieved evidence with learned or prompted behavior. Here the model receives both policy context and style examples, then returns a structured comparison that supports an architecture recommendation.


## Learning Objectives

By the end of this notebook, you should be able to:

- Define freshness, auditability, consistency, cost, and latency criteria.
- Build a hybrid prompt from retrieved policies and style examples.
- Compare RAG-only, style-only, and hybrid answers with one rubric.
- Write a decision report with a trigger for fine-tuning.


## What A Strategy Decision Must Contain

A useful architecture decision is not just a winner label. It should say what evidence was tested, which risks remain, and what trigger would justify revisiting the decision. For RAG and fine-tuning choices, the most important risks are freshness, citation auditability, behavior consistency, cost, latency, and maintenance effort.

The hybrid strategy is often attractive, but it should not be the default just because it sounds complete. Start with the simplest approach that satisfies the evaluation criteria. Add style conditioning or fine-tuning only when the measured failures point to behavior adaptation rather than missing evidence.


## Decision Criteria Definitions

**Freshness** means the system can reflect updated policy without retraining. RAG usually wins here because documents can be updated and re-indexed. **Auditability** means a reviewer can trace an answer back to evidence. Citations and source IDs support this. **Behavior consistency** means the system responds with the same tone, format, and refusal style across similar requests. Fine-tuning or strong examples can help here.

**Latency** is the time needed to answer. Retrieval and larger prompts can add overhead. **Cost** includes provider calls, embeddings, vector storage, training, and maintenance. **Maintenance effort** includes keeping documents current, reviewing examples, monitoring failures, and deciding when a strategy should be revisited.


## Does Combining Strategies Always Help?

The decision report compares retrieval quality, behavior consistency, operational cost, and the trigger for a future fine-tuning experiment. A hybrid sounds like the safest choice, but measurements do not always support it.

The chart below comes from a study that added new knowledge to three 7B models with fine-tuning (FT), RAG, or both. RAG gave a large gain on every model, fine-tuning alone helped little, and combining them beat RAG alone only for Llama2. That study combined RAG with fine-tuning, while this notebook combines RAG with style examples, but the lesson carries over: treat a hybrid as a hypothesis to measure, not a default.

<div align="center">
  <img src="assets/05-knowledge-injection-gains.png" alt="Mean accuracy gain of fine-tuning, RAG, and both across three models" width="600" style="max-width: 100%; height: auto;" />
  <p><em>Mean accuracy gain from fine-tuning (FT), RAG, and both (FT+RAG) on knowledge-injection tasks. Source: <a href="https://arxiv.org/abs/2312.05934" target="_blank" rel="noopener noreferrer">Ovadia et al., 2023</a>, CC BY 4.0</em></p>
</div>


In [ ]:
import json
import re
from pprint import pprint
from typing import Any

from pydantic import BaseModel, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    ModelReply,
    ReplySource,
    call_model,
    format_context,
    retrieve,
    style_messages,
)

## Load Shared Data And Runtime Helpers

All notebooks use the same fixtures and provider gateway so the comparisons stay controlled. The repeated setup lives in [`strategy_helpers.py`](strategy_helpers.py); lesson cells can now focus on the architecture decision being tested. Dry mode remains deterministic when no provider key is available.


In [ ]:
print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

## Reuse The Evaluated Retrieval Baseline

The hybrid comparison reuses the same ranked-search helper as the RAG-only notebook. Keeping retrieval fixed means later differences come from behavior conditioning rather than a different evidence path.


In [ ]:
retrieval_preview = retrieve(REQUESTS[0]["question"], top_k=3)
[(row["doc_id"], round(row["score"], 3)) for row in retrieval_preview]

## Reuse Approved Style Examples

The shared helper converts approved examples into alternating user and assistant messages. This keeps the behavior signal identical to the fine-tuning-style baseline.


In [ ]:
few_shot_preview = style_messages(STYLE_EXAMPLES[:2])
[(message["role"], message["content"][:60]) for message in few_shot_preview]

## Build The Hybrid Prompt

The hybrid prompt includes the top retrieved policy records plus examples of the desired response behavior. This is the pattern to compare against RAG-only and style-only prompts.


Use the same answer schema for RAG-only, style-only, and hybrid prompts. Execution provenance is supplied by the application, never trusted from model output. Scores are calculated locally below rather than requested as model self-ratings.

In [ ]:
class StrategyResult(BaseModel):
    """One answer plus application-owned execution and parsing metadata."""

    answer: str
    source_ids: list[str]
    response_source: ReplySource = "simulated"
    parse_valid: bool = True

Keep one prompt builder and vary only the presence of retrieved context and style examples. Every condition returns JSON with `answer` and `source_ids`.

In [ ]:
def strategy_messages(
    request: dict[str, Any], evidence: list[dict[str, Any]], strategy: str
) -> list[dict[str, str]]:
    """Keep output instructions identical across the three prompting conditions."""
    examples = style_messages(STYLE_EXAMPLES) if strategy != "rag" else []
    return [
        {
            "role": "system",
            "content": "Answer only from supplied policy context. If it is insufficient, say so. Return JSON with answer and source_ids; cite policy IDs in square brackets. Do not invent IDs.",
        },
        *examples,
        {
            "role": "user",
            "content": f"Context:\n{format_context(evidence) or 'No policy context supplied.'}\n\nQuestion: {request['question']}",
        },
    ]

Without a live response, use an explicitly labeled illustrative answer. It never reads the expected evaluation label to manufacture a successful score.

In [ ]:
def simulated_response(evidence: list[dict[str, Any]]) -> str:
    """Supply a transparent example, not evidence of model performance."""
    if not evidence:
        return json.dumps(
            {"answer": "I need current policy evidence to answer.", "source_ids": []}
        )
    first = evidence[0]
    return json.dumps(
        {
            "answer": f"{first['text']} [{first['doc_id']}]",
            "source_ids": [first["doc_id"]],
        }
    )

Parse the reply and preserve whether it was live, simulated, or a fallback. Malformed JSON is a failed format check, not an answer to score as successful.

In [ ]:
def strategy_answer(request: dict[str, Any], strategy: str) -> StrategyResult:
    """Run one condition against the same query and output contract."""
    evidence = (
        [] if strategy == "style_only" else retrieve(request["question"], top_k=3)
    )
    reply = call_model(
        strategy_messages(request, evidence, strategy),
        dry_response=simulated_response(evidence),
        response_format={"type": "json_object"},
    )
    try:
        result = StrategyResult.model_validate_json(reply.text)
        result.response_source = reply.source
        result.parse_valid = True
        return result
    except ValidationError:
        return StrategyResult(
            answer="", source_ids=[], response_source=reply.source, parse_valid=False
        )

Run one hybrid call before building the strategy table so the output shape is visible.


In [ ]:
result = strategy_answer(REQUESTS[2], "hybrid")
pprint(result.model_dump())

## Compare All Three Conditions With One Rubric

For each held-out request, run RAG-only, style-only, and hybrid prompting. The common checks measure parsing, retrieval coverage, citation validity, expected-source citation, and response length. None is a semantic faithfulness or tone judge.

## Interpreting The Comparison Table

`retrieval_hit` checks whether the expected policy entered the context. `citations_valid` requires matching inline and declared citations drawn from that context. `expected_source_cited` checks whether the labeled policy was cited. `concise` only checks a maximum of 80 words, not tone or correctness.

Inspect the actual answers too. A correct source ID does not prove its claims. Rows marked `simulated` or `fallback` demonstrate the rubric, but are excluded from the live architecture recommendation.

## Example Strategy Recommendations

If retrieval grounding is high and style is acceptable, recommend RAG-only because it is simpler and easier to maintain. If retrieval grounding is high but tone is inconsistent, recommend a hybrid prompt with better examples before scheduling fine-tuning. If retrieval grounding is low, recommend improving chunking, query rewriting, or the corpus before changing model behavior.

Fine-tuning becomes a stronger recommendation only after repeated behavior failures are measured on held-out examples. Even then, it should be compared against a stronger prompt baseline. The decision report should explain what was tried, why the chosen path is sufficient for now, and what signal would trigger the next experiment.


In [ ]:
def score_answer(
    request: dict[str, Any], strategy: str, result: StrategyResult
) -> dict[str, Any]:
    """Apply identical observable checks without asking the model to grade itself."""
    evidence = (
        [] if strategy == "style_only" else retrieve(request["question"], top_k=3)
    )
    retrieved_ids = {item["doc_id"] for item in evidence}
    inline_ids = set(re.findall(r"\[([^\[\]]*)\]", result.answer))
    declared_ids = set(result.source_ids)
    valid = (
        result.parse_valid
        and bool(inline_ids)
        and inline_ids == declared_ids
        and inline_ids.issubset(retrieved_ids)
    )
    return {
        "request_id": request["request_id"],
        "strategy": strategy,
        "response_source": result.response_source,
        "answer": result.answer,
        "format_valid": result.parse_valid,
        "retrieval_hit": request["expected_doc"] in retrieved_ids,
        "citations_valid": valid,
        "expected_source_cited": valid and request["expected_doc"] in declared_ids,
        "concise": bool(result.answer.strip()) and len(result.answer.split()) <= 80,
    }


strategy_rows = [
    score_answer(request, strategy, strategy_answer(request, strategy))
    for request in REQUESTS
    for strategy in ("rag", "style_only", "hybrid")
]
pprint(strategy_rows)

### Exercise: Summarize Live Rows Per Strategy

The decision report may only use live answers. For each strategy, count the live rows in `strategy_rows` and how many of them cite the expected source. Skip rows whose `response_source` is `simulated` or `fallback`.


In [ ]:
# @TODO: Count live rows and expected-source citations per strategy.


<details>
<summary>Show Reference Solution</summary>

```python
live_summary = {}
for row in strategy_rows:
    if row["response_source"] != "live":
        continue
    counts = live_summary.setdefault(
        row["strategy"], {"live_rows": 0, "expected_source_cited": 0}
    )
    counts["live_rows"] += 1
    counts["expected_source_cited"] += row["expected_source_cited"]
live_summary
```

</details>


Filtering changes the picture. Simulated and fallback rows reuse a deterministic answer that copies the top retrieved document with a valid citation, so they often pass every check. Counting them would make a strategy look better than the live model performed. If a strategy has few or no live rows, for example after rate-limit errors, the report should say that it was not measured rather than pick a winner.


## Produce The Decision Report

Only a complete live comparison can support the model-based recommendation below. Otherwise, report that no live comparison was measured. Few-shot prompting still does not establish whether trained LoRA weights would help.

Build decision-review messages separately from the dry response so the report structure is visible.


In [ ]:
live_comparison_complete = all(
    row["response_source"] == "live" for row in strategy_rows
)
decision_messages = [
    {
        "role": "system",
        "content": "Review these common-rubric results and actual answers. Return JSON with recommended_strategy, rationale, and revisit_trigger. Citation validity does not prove faithfulness; concision does not prove tone. These are prompts, not trained fine-tuned models.",
    },
    {"role": "user", "content": json.dumps(strategy_rows, indent=2)},
]

If any answer was simulated or fell back after an API error, do not present the comparison as a measured live experiment.

In [ ]:
dry_decision = json.dumps(
    {
        "recommended_strategy": "not evaluated",
        "rationale": "A complete live comparison was not measured; simulated and fallback answers are illustrative only.",
        "revisit_trigger": "Run the same rubric on live answers and inspect policy claims before choosing a strategy.",
    },
    indent=2,
)

Call the reviewer only after a complete live comparison; otherwise show the explicit no-measurement report without an extra provider call.

In [ ]:
decision_reply = (
    call_model(
        decision_messages,
        dry_response=dry_decision,
        response_format={"type": "json_object"},
    )
    if live_comparison_complete
    else ModelReply(text=dry_decision, source="simulated")
)
print(decision_reply.source, decision_reply.text)

## Check Your Understanding

1. Which risk does citation auditability address: freshness, grounding, or tone?
2. What two signals does the hybrid strategy combine?
3. What evidence would justify moving from prompt improvements to real fine-tuning?

<details>
<summary>Show Reference Answers</summary>

1. Citation auditability mainly addresses grounding because reviewers can trace an answer back to retrieved evidence.
2. It combines retrieved evidence for grounding with style examples for response behavior.
3. Repeated style or formatting failures on held-out evaluation examples would justify a real fine-tuning experiment.

</details>


## Further Reading

- [RAFT: Adapting Language Model to Domain Specific RAG](https://arxiv.org/abs/2403.10131) - Zhang et al., 2024
